# N02 Main Evaluation — Dataset Design & Preflight v1.0

**Goal:** freeze a controlled Main Evaluation corpus for the three N02 core observables: length-adjusted block-update norm, input–update cosine, and output/input norm ratio.

## Study design
- **100 semantic topic clusters × 4 matched text styles = 400 candidate texts.**
- Styles: general prose, question/dialogue, academic prose, and code + natural language.
- Length tiers: 34 short clusters, 33 medium clusters, 33 long clusters. Candidate token bands are 16–64, 65–128, and 129–256 tokens, respectively, and must be checked separately with each model tokenizer.
- Each `topic_id` has one rendition in each style. This matches content at a topic level, but the four renditions are repeated/matched observations, not four independent semantic units.
- Therefore, the planned confidence intervals for style contrasts resample **topic_id clusters** (10,000 bootstrap replicates), preserving all four renditions. Do not resample the 400 rows independently for matched-style contrasts.
- Pilot prompts are explicitly excluded.

**Preflight only:** this notebook audits the candidate text manifest. It does not load the language models or capture activations. No model activations may be examined until the final text manifest has passed the tokenizer/quality checks and its SHA-256 hash has been recorded.

The corpus is template-assisted and should receive human content/quality review. The code-mixed condition is a distinct format condition, not a pure isolated linguistic style; interpretation must acknowledge that code syntax itself differs from prose.


In [1]:
%pip -q install -U transformers pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 132.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 150.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 136.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.


## 1. Upload the candidate manifest

Download `N02_Main_Evaluation_Candidate_Manifest_v1.0.csv` with this notebook, then upload it when prompted. This is a **candidate** manifest, not yet frozen for confirmatory analysis.


In [2]:
from pathlib import Path
import os, json, hashlib, re, math
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

manifest_path = Path('/content/N02_Main_Evaluation_Candidate_Manifest_v1.0.csv')
if not manifest_path.exists():
    from google.colab import files
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
    if not csv_names:
        raise FileNotFoundError('Upload N02_Main_Evaluation_Candidate_Manifest_v1.0.csv')
    manifest_path = Path('/content') / csv_names[0]

candidate_df = pd.read_csv(manifest_path)
print('Manifest:', manifest_path)
print('Rows:', len(candidate_df), 'Unique topic clusters:', candidate_df['topic_id'].nunique())
display(candidate_df.head(8))

Saving N02_Main_Evaluation_Candidate_Manifest_v1.0.csv to N02_Main_Evaluation_Candidate_Manifest_v1.0.csv
Manifest: /content/N02_Main_Evaluation_Candidate_Manifest_v1.0.csv
Rows: 400 Unique topic clusters: 100


,topic_id,topic_domain,operation_id,length_tier,style,text_id,text,role,pilot_excluded,sampling_unit,generation_seed
0,T001,turbine_vibration,filter_threshold,short,general_prose,T001_general_prose,A script must retain only the vibration amplit...,candidate_main_eval_not_frozen,True,topic_id_cluster,20261011
1,T001,turbine_vibration,filter_threshold,short,question_dialogue,T001_question_dialogue,A technician is reviewing the turbine measurem...,candidate_main_eval_not_frozen,True,topic_id_cluster,20261011
2,T001,turbine_vibration,filter_threshold,short,academic,T001_academic,The preprocessing step is specified to retain ...,candidate_main_eval_not_frozen,True,topic_id_cluster,20261011
3,T001,turbine_vibration,filter_threshold,short,code_plus_natural_language,T001_code_plus_natural_language,"values = [2.4, 3.1, 1.8, 4.0]\nresult = [v for...",candidate_main_eval_not_frozen,True,topic_id_cluster,20261011
4,T002,network_delay,minmax_normalize,short,general_prose,T002_general_prose,A script must scale the packet-delay measureme...,candidate_main_eval_not_frozen,True,topic_id_cluster,20261011
5,T002,network_delay,minmax_normalize,short,question_dialogue,T002_question_dialogue,A technician is reviewing the network delay me...,candidate_main_eval_not_frozen,True,topic_id_cluster,20261011
6,T002,network_delay,minmax_normalize,short,academic,T002_academic,The preprocessing step is specified to scale t...,candidate_main_eval_not_frozen,True,topic_id_cluster,20261011
7,T002,network_delay,minmax_normalize,short,code_plus_natural_language,T002_code_plus_natural_language,"values = [8, 14, 11, 23]\nlo, hi = min(values)...",candidate_main_eval_not_frozen,True,topic_id_cluster,20261011


## 2. Structural validation

A valid candidate must contain 100 topic clusters, exactly 4 unique styles per cluster, 400 rows total, no duplicate `text_id`, and no duplicated normalized text. These checks are necessary, not sufficient; content quality and token counts still need review.


In [3]:
EXPECTED_STYLES = {
    'general_prose', 'question_dialogue', 'academic', 'code_plus_natural_language'
}
EXPECTED_TIER_COUNTS = {'short': 34, 'medium': 33, 'long': 33}
LENGTH_BANDS = {'short': (16, 64), 'medium': (65, 128), 'long': (129, 256)}

required_columns = {
    'topic_id', 'topic_domain', 'operation_id', 'length_tier', 'style',
    'text_id', 'text', 'role', 'pilot_excluded', 'sampling_unit'
}
missing_columns = required_columns - set(candidate_df.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

checks = {
    'rows_400': len(candidate_df) == 400,
    'topics_100': candidate_df['topic_id'].nunique() == 100,
    'text_id_unique': candidate_df['text_id'].is_unique,
    'nonempty_texts': candidate_df['text'].fillna('').str.strip().ne('').all(),
    'styles_exact': set(candidate_df['style']) == EXPECTED_STYLES,
    'no_duplicate_normalized_text': not candidate_df['text'].fillna('').str.lower().str.replace(r'\s+', ' ', regex=True).str.strip().duplicated().any(),
    'all_marked_pilot_excluded': candidate_df['pilot_excluded'].astype(str).str.lower().isin(['true','1']).all(),
}
cluster_style_counts = candidate_df.groupby('topic_id')['style'].nunique()
checks['four_styles_per_topic'] = len(cluster_style_counts) == 100 and cluster_style_counts.eq(4).all()
tier_topic_counts = candidate_df.drop_duplicates('topic_id')['length_tier'].value_counts().to_dict()
checks['tier_topic_counts_34_33_33'] = tier_topic_counts == EXPECTED_TIER_COUNTS

print('Structural checks:')
for name, passed in checks.items(): print(('PASS' if passed else 'FAIL'), name)
if not all(checks.values()):
    raise ValueError('Structural validation failed. Fix candidate manifest before tokenizer preflight.')
display(candidate_df.groupby(['length_tier','style']).size().unstack(fill_value=0))

Structural checks:
PASS rows_400
PASS topics_100
PASS text_id_unique
PASS nonempty_texts
PASS styles_exact
PASS no_duplicate_normalized_text
PASS all_marked_pilot_excluded
PASS four_styles_per_topic
PASS tier_topic_counts_34_33_33


style,academic,code_plus_natural_language,general_prose,question_dialogue
length_tier,,,,
long,33,33,33,33
medium,33,33,33,33
short,34,34,34,34


## 3. Tokenizer-specific length audit

Run exact token counts separately for the two current N02 pilot model tokenizers: GPT-2 and Pythia-160M. A text only passes this preflight if its token count falls inside the assigned tier for **both** tokenizers. Count includes tokenizer-added special tokens as used in the audit below. Do not inspect activation values during text revision.


In [4]:
from transformers import AutoTokenizer

TOKENIZER_IDS = {
    'gpt2': 'gpt2',
    'pythia-160m': 'EleutherAI/pythia-160m',
}
tokenizers = {key: AutoTokenizer.from_pretrained(model_id) for key, model_id in TOKENIZER_IDS.items()}

for model_key, tokenizer in tokenizers.items():
    candidate_df[f'tokens_{model_key}'] = candidate_df['text'].map(
        lambda text: len(tokenizer(text, add_special_tokens=True, truncation=False)['input_ids'])
    )
    low_high = candidate_df['length_tier'].map(LENGTH_BANDS)
    candidate_df[f'in_band_{model_key}'] = [
        lo <= int(n) <= hi for (lo, hi), n in zip(low_high, candidate_df[f'tokens_{model_key}'])
    ]

for model_key in TOKENIZER_IDS:
    col = f'tokens_{model_key}'
    ok_col = f'in_band_{model_key}'
    print(f'\n{model_key}: in-band {int(candidate_df[ok_col].sum())}/{len(candidate_df)}')
    display(candidate_df.loc[~candidate_df[ok_col], ['topic_id','length_tier','style','text_id',col,'text']].head(25))

length_summary = candidate_df.groupby(['length_tier','style']).agg(
    count=('text_id','count'),
    gpt2_median=('tokens_gpt2','median'),
    gpt2_min=('tokens_gpt2','min'),
    gpt2_max=('tokens_gpt2','max'),
    pythia_median=('tokens_pythia-160m','median'),
    pythia_min=('tokens_pythia-160m','min'),
    pythia_max=('tokens_pythia-160m','max'),
).reset_index()
display(length_summary)

# Topic-level within-matched-set spread; warnings are for pre-activation revision.
spread_rows = []
for topic_id, group in candidate_df.groupby('topic_id'):
    row = {'topic_id': topic_id, 'length_tier': group['length_tier'].iloc[0]}
    for model_key in TOKENIZER_IDS:
        counts = group[f'tokens_{model_key}'].astype(float)
        row[f'{model_key}_token_min'] = counts.min()
        row[f'{model_key}_token_max'] = counts.max()
        row[f'{model_key}_token_ratio_max_min'] = counts.max() / max(counts.min(), 1)
        row[f'{model_key}_token_spread'] = counts.max() - counts.min()
    spread_rows.append(row)
spread_df = pd.DataFrame(spread_rows)
print('Topic groups with >1.8x max/min token spread (review before freeze):')
display(spread_df[(spread_df['gpt2_token_ratio_max_min'] > 1.8) | (spread_df['pythia-160m_token_ratio_max_min'] > 1.8)].head(30))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...


pythia-160m: in-band 393/400


,topic_id,length_tier,style,text_id,tokens_pythia-160m,text
7,T002,short,code_plus_natural_language,T002_code_plus_natural_language,71,"values = [8, 14, 11, 23]\nlo, hi = min(values)..."
27,T007,short,code_plus_natural_language,T007_code_plus_natural_language,71,"values = [81, 75, 120, 95]\nlo, hi = min(value..."
127,T032,short,code_plus_natural_language,T032_code_plus_natural_language,78,"values = [2.4, 3.1, 1.8, 4.0]\nlo, hi = min(va..."
155,T039,medium,code_plus_natural_language,T039_code_plus_natural_language,145,"def process(values):\n """"""Apply one documen..."
163,T041,medium,code_plus_natural_language,T041_code_plus_natural_language,146,"def process(values):\n """"""Apply one documen..."
191,T048,medium,code_plus_natural_language,T048_code_plus_natural_language,139,"def process(values):\n """"""Apply one documen..."
267,T067,medium,code_plus_natural_language,T067_code_plus_natural_language,138,"def process(values):\n """"""Apply one documen..."


,length_tier,style,count,gpt2_median,gpt2_min,gpt2_max,pythia_median,pythia_min,pythia_max
0,long,academic,33,157.0,152,171,157.0,152,170
1,long,code_plus_natural_language,33,229.0,209,302,205.0,186,255
2,long,general_prose,33,165.0,160,179,163.0,158,176
3,long,question_dialogue,33,137.0,130,148,136.0,129,146
4,medium,academic,33,107.0,99,117,106.0,98,115
5,medium,code_plus_natural_language,33,129.0,115,177,108.0,95,146
6,medium,general_prose,33,91.0,81,98,91.0,81,97
7,medium,question_dialogue,33,89.0,80,97,88.0,80,96
8,short,academic,34,40.0,35,47,39.0,34,45
9,short,code_plus_natural_language,34,48.0,38,78,49.0,39,78


Topic groups with >1.8x max/min token spread (review before freeze):


,topic_id,length_tier,gpt2_token_min,gpt2_token_max,gpt2_token_ratio_max_min,gpt2_token_spread,pythia-160m_token_min,pythia-160m_token_max,pythia-160m_token_ratio_max_min,pythia-160m_token_spread
31,T032,short,41.0,78.0,1.902439,37.0,39.0,78.0,2.000000,39.0
38,T039,medium,91.0,176.0,1.934066,85.0,90.0,145.0,1.611111,55.0
40,T041,medium,97.0,177.0,1.824742,80.0,96.0,146.0,1.520833,50.0
84,T085,long,146.0,300.0,2.054795,154.0,142.0,251.0,1.767606,109.0
87,T088,long,144.0,290.0,2.013889,146.0,142.0,243.0,1.711268,101.0
89,T090,long,148.0,302.0,2.040541,154.0,146.0,255.0,1.746575,109.0


## 4. Revision gate and freeze

Before any activation capture:
1. Review text quality and semantic matching across each four-style topic cluster.
2. Revise any text outside its assigned token band using only prompt text/tokenizer counts (do not look at model activations first).
3. Inspect topic-level token spread and the length summaries; match/revise texts with large within-topic spread where feasible.
4. Save the finalized manifest and hash it. The confirmatory run must read the frozen file and not silently regenerate or edit it.

The cell below **will not declare the manifest frozen if any candidate is outside its assigned tier for either tokenizer**. It creates a preflight report regardless. The semantic/content review checkbox is an explicit human attestation; leave it `False` until that review is done.


In [6]:
CONTENT_REVIEW_COMPLETE = False  # Change to True only after human review of all topic-matched quartets.

band_failures = int((~candidate_df['in_band_gpt2']).sum() + (~candidate_df['in_band_pythia-160m']).sum())
all_bands_pass = candidate_df['in_band_gpt2'].all() and candidate_df['in_band_pythia-160m'].all()
all_styles_per_topic = candidate_df.groupby('topic_id')['style'].nunique().eq(4).all()
freeze_allowed = bool(all(checks.values()) and all_bands_pass and all_styles_per_topic and CONTENT_REVIEW_COMPLETE)

# Always write diagnostics, even when freeze is blocked.
OUT_DIR = Path('/content/n02_main_eval_preflight')
OUT_DIR.mkdir(parents=True, exist_ok=True)
candidate_df.to_csv(OUT_DIR / 'N02_candidate_manifest_with_token_counts.csv', index=False)
length_summary.to_csv(OUT_DIR / 'N02_token_length_summary.csv', index=False)
spread_df.to_csv(OUT_DIR / 'N02_topic_token_spread.csv', index=False)

# Fix: Use .str.cat(sep='\n') to combine the series into a single string before encoding
manifest_sha256 = hashlib.sha256(candidate_df['text_id'].astype(str).str.cat(candidate_df['text'].astype(str), sep='\n').str.cat(sep='\n').encode('utf-8')).hexdigest()
report = {
    'protocol_version': 'N02_Main_Evaluation_Dataset_Design_v1.0',
    'rows': int(len(candidate_df)),
    'topic_clusters': int(candidate_df['topic_id'].nunique()),
    'styles_per_topic': 4,
    'tokenizers': TOKENIZER_IDS,
    'length_bands': LENGTH_BANDS,
    'band_failures_across_tokenizers': band_failures,
    'all_candidates_in_band_for_both_tokenizers': bool(all_bands_pass),
    'human_content_review_complete': bool(CONTENT_REVIEW_COMPLETE),
    'frozen_for_confirmatory_analysis': freeze_allowed,
    'candidate_text_manifest_sha256': manifest_sha256,
    'resampling_unit_for_matched_style_contrasts': 'topic_id cluster; retain all four styles per resampled topic',
    'bootstrap_replicates': 10000,
    'pilot_prompts_excluded': True,
    'notes': 'Do not capture activations unless frozen_for_confirmatory_analysis is true. The code-mixed condition is a format condition as well as a style condition.'
}
with open(OUT_DIR / 'N02_preflight_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2, ensure_ascii=False))

if freeze_allowed:
    frozen_path = OUT_DIR / 'N02_Main_Evaluation_FROZEN_Manifest.csv'
    candidate_df.to_csv(frozen_path, index=False)
    frozen_file_sha256 = hashlib.sha256(frozen_path.read_bytes()).hexdigest()
    report['frozen_manifest_file_sha256'] = frozen_file_sha256
    report['frozen_manifest_path'] = str(frozen_path)
    with open(OUT_DIR / 'N02_preflight_report.json', 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print('FROZEN MANIFEST:', frozen_path)
    print('FROZEN FILE SHA256:', frozen_file_sha256)
else:
    print('NOT FROZEN: resolve token-band failures and complete content review before activation capture.')

{
  "protocol_version": "N02_Main_Evaluation_Dataset_Design_v1.0",
  "rows": 400,
  "topic_clusters": 100,
  "styles_per_topic": 4,
  "tokenizers": {
    "gpt2": "gpt2",
    "pythia-160m": "EleutherAI/pythia-160m"
  },
  "length_bands": {
    "short": [
      16,
      64
    ],
    "medium": [
      65,
      128
    ],
    "long": [
      129,
      256
    ]
  },
  "band_failures_across_tokenizers": 30,
  "all_candidates_in_band_for_both_tokenizers": false,
  "human_content_review_complete": false,
  "frozen_for_confirmatory_analysis": false,
  "candidate_text_manifest_sha256": "dbdb2874609e13dbc7769b5ea25d8f43e34179547e0f0dda87b3475a54231843",
  "resampling_unit_for_matched_style_contrasts": "topic_id cluster; retain all four styles per resampled topic",
  "bootstrap_replicates": 10000,
  "pilot_prompts_excluded": true,
  "notes": "Do not capture activations unless frozen_for_confirmatory_analysis is true. The code-mixed condition is a format condition as well as a style condition.

In [7]:
failed_mask = (~candidate_df['in_band_gpt2']) | (~candidate_df['in_band_pythia-160m'])
failed_df = candidate_df[failed_mask]

print(f"총 실패 항목 수: {len(failed_df)}\n")

# 스타일별 실패 항목 수
print("=== 스타일(Style)별 실패 분포 ===")
style_counts = failed_df['style'].value_counts()
display(style_counts.to_frame(name='실패 수'))

# 길이 대역(Length Tier)별 실패 항목 수
print("\n=== 길이 대역(Length Tier)별 실패 분포 ===")
tier_counts = failed_df['length_tier'].value_counts()
display(tier_counts.to_frame(name='실패 수'))

# 스타일 및 길이 대역 교차 요약
print("\n=== 스타일 x 길이 대역 교차 요약 ===")
cross_tab = pd.crosstab(failed_df['style'], failed_df['length_tier'], margins=True)
display(cross_tab)

# 실패한 구체적인 topic_id와 스타일 리스트 출력 (상위 30개 전체)
print("\n=== 실패한 상세 항목 리스트 (Topic ID 및 Style) ===")
display(failed_df[['topic_id', 'length_tier', 'style', 'tokens_gpt2', 'tokens_pythia-160m']].sort_values(by='topic_id'))

총 실패 항목 수: 23

=== 스타일(Style)별 실패 분포 ===


,실패 수
style,
code_plus_natural_language,23



=== 길이 대역(Length Tier)별 실패 분포 ===


,실패 수
length_tier,
medium,17
short,3
long,3



=== 스타일 x 길이 대역 교차 요약 ===


length_tier,long,medium,short,All
style,,,,
code_plus_natural_language,3,17,3,23
All,3,17,3,23



=== 실패한 상세 항목 리스트 (Topic ID 및 Style) ===


,topic_id,length_tier,style,tokens_gpt2,tokens_pythia-160m
7,T002,short,code_plus_natural_language,71,71
27,T007,short,code_plus_natural_language,71,71
127,T032,short,code_plus_natural_language,78,78
139,T035,medium,code_plus_natural_language,142,122
147,T037,medium,code_plus_natural_language,129,108
155,T039,medium,code_plus_natural_language,176,145
159,T040,medium,code_plus_natural_language,142,122
163,T041,medium,code_plus_natural_language,177,146
167,T042,medium,code_plus_natural_language,135,115
171,T043,medium,code_plus_natural_language,142,122


In [8]:
# GPT-2나 Pythia 기준 목표 대역의 최댓값을 초과하거나 최솟값에 한참 미달하는 아웃라이어들을 식별하기 위해,
# 각 길이 대역별 기준(short: 16-64, medium: 65-128, long: 129-256) 대비 토큰 수 차이의 절대값이 큰 상위 항목들을 출력합니다.

def get_deviation(row):
    low, high = LENGTH_BANDS[row['length_tier']]
    # 양쪽 토크나이저 중 더 많이 벗어난 정도를 계산
    dev_gpt2 = max(0, row['tokens_gpt2'] - high) + max(0, low - row['tokens_gpt2'])
    dev_pythia = max(0, row['tokens_pythia-160m'] - high) + max(0, low - row['tokens_pythia-160m'])
    return max(dev_gpt2, dev_pythia)

failed_df_with_dev = failed_df.copy()
failed_df_with_dev['deviation'] = failed_df_with_dev.apply(get_deviation, axis=1)

# 가장 많이 벗어난 상위 5개 토픽 선정
top_failed = failed_df_with_dev.sort_values(by='deviation', ascending=False).head(5)

for idx, row in top_failed.iterrows():
    low, high = LENGTH_BANDS[row['length_tier']]
    print(f"=========================================================================")
    print(f"[Topic ID: {row['topic_id']}] | Domain: {row['topic_domain']} | Style: {row['style']}")
    print(f"Target Band: {row['length_tier']} ({low}~{high} tokens)")
    print(f"Actual Tokens -> GPT-2: {row['tokens_gpt2']} | Pythia-160M: {row['tokens_pythia-160m']}")
    print(f"=========================================================================")
    print(row['text'])
    print("\n")

[Topic ID: T041] | Domain: solar_output | Style: code_plus_natural_language
Target Band: medium (65~128 tokens)
Actual Tokens -> GPT-2: 177 | Pythia-160M: 146
def process(values):
    """Apply one documented transformation to a sample."""
    if not values:
        return None
    if any(value is None for value in values):
        raise ValueError("Missing value")
    low, high = min(values), max(values)
    if high == low:
        return [0.0 for _ in values]
    return [(value-low)/(high-low) for value in values]

sample = [2.1, 3.4, 2.9, 4.0]
result = process(sample)
print(result)
# Explain how this output should be interpreted for solar output readings.


[Topic ID: T039] | Domain: greenhouse_temperature | Style: code_plus_natural_language
Target Band: medium (65~128 tokens)
Actual Tokens -> GPT-2: 176 | Pythia-160M: 145
def process(values):
    """Apply one documented transformation to a sample."""
    if not values:
        return None
    if any(value is None for value in values

In [9]:
import pandas as pd

# 각 토픽별로 Docstring과 예외 처리를 제거한 축소본 텍스트 정의
modified_texts = {
    'T041': """def process(values):
    if not values:
        return None
    low, high = min(values), max(values)
    if high == low:
        return [0.0 for _ in values]
    return [(value-low)/(high-low) for value in values]

sample = [2.1, 3.4, 2.9, 4.0]
result = process(sample)
print(result)
# Explain how this output should be interpreted for solar output readings.""",

    'T039': """def process(values):
    if not values:
        return None
    low, high = min(values), max(values)
    if high == low:
        return [0.0 for _ in values]
    return [(value-low)/(high-low) for value in values]

sample = [18.4, 19.1, 17.8, 20.0]
result = process(sample)
print(result)
# Explain how this output should be interpreted for greenhouse readings.""",

    'T048': """def process(values):
    if not values:
        return None
    low, high = min(values), max(values)
    if high == low:
        return [0.0 for _ in values]
    return [(value-low)/(high-low) for value in values]

sample = [205, 198, 212, 220]
result = process(sample)
print(result)
# Explain how this output should be interpreted for coolant pressure samples."""
}

# 토크나이저 검증
for topic_id, text in modified_texts.items():
    gpt2_tokens = len(tokenizers['gpt2'](text, add_special_tokens=True)['input_ids'])
    pythia_tokens = len(tokenizers['pythia-160m'](text, add_special_tokens=True)['input_ids'])

    print(f"=== [Topic ID: {topic_id}] 수정 후 검증 ===")
    print(f"Target Band: medium (65~128 tokens)")
    print(f"Actual Tokens -> GPT-2: {gpt2_tokens} | Pythia-160M: {pythia_tokens}")
    print("Status: ", "PASS (In-Band)" if (65 <= gpt2_tokens <= 128 and 65 <= pythia_tokens <= 128) else "FAIL")
    print("--- Modified Text ---")
    print(text)
    print("\n")

=== [Topic ID: T041] 수정 후 검증 ===
Target Band: medium (65~128 tokens)
Actual Tokens -> GPT-2: 133 | Pythia-160M: 113
Status:  FAIL
--- Modified Text ---
def process(values):
    if not values:
        return None
    low, high = min(values), max(values)
    if high == low:
        return [0.0 for _ in values]
    return [(value-low)/(high-low) for value in values]

sample = [2.1, 3.4, 2.9, 4.0]
result = process(sample)
print(result)
# Explain how this output should be interpreted for solar output readings.


=== [Topic ID: T039] 수정 후 검증 ===
Target Band: medium (65~128 tokens)
Actual Tokens -> GPT-2: 132 | Pythia-160M: 112
Status:  FAIL
--- Modified Text ---
def process(values):
    if not values:
        return None
    low, high = min(values), max(values)
    if high == low:
        return [0.0 for _ in values]
    return [(value-low)/(high-low) for value in values]

sample = [18.4, 19.1, 17.8, 20.0]
result = process(sample)
print(result)
# Explain how this output should be interpreted

In [10]:
import re

def auto_compress_text(text, style):
    # 이 압축 규칙은 주로 문제가 된 code_plus_natural_language 스타일에만 적용합니다.
    if style != 'code_plus_natural_language':
        return text

    # 1. Docstring 제거 (""" ... """)
    text = re.sub(r'"""[\s\S]*?"""', '', text)
    text = re.sub(r"'''[\s\S]*?'''", '', text)

    # 2. raise ValueError/Exception 구문이 있는 줄 삭제
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        if re.search(r'\braise\s+(ValueError|Exception|TypeError)', line):
            continue
        cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)

    # 3. 맨 아래에 있는 길고 장황한 주석 단순화
    # 예: # Explain how this output should be interpreted for... -> # Interpret output.
    text = re.sub(r'#\s*Explain\s+how\s+this\s+output\s+should\s+be\s+interpreted.*', '# Interpret output.', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*Context:.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*Task:.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*Interpretation:.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*Validation:.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*Explain\s+whether.*', '', text, flags=re.IGNORECASE)

    # 4. 연속된 공백 라인 축소
    text = re.sub(r'\n\s*\n+', '\n\n', text).strip()

    return text

# 실패한 23개 항목이 들어있는 candidate_df의 복사본 생성
compressed_df = candidate_df.copy()

# 실패한 인덱스에 대해 자동 압축 함수 적용
for idx in failed_df.index:
    original_text = compressed_df.loc[idx, 'text']
    style = compressed_df.loc[idx, 'style']
    compressed_df.loc[idx, 'text'] = auto_compress_text(original_text, style)

# 토큰 수 재계산
for model_key, tokenizer in tokenizers.items():
    compressed_df[f'tokens_{model_key}'] = compressed_df['text'].map(
        lambda text: len(tokenizer(text, add_special_tokens=True, truncation=False)['input_ids'])
    )
    low_high = compressed_df['length_tier'].map(LENGTH_BANDS)
    compressed_df[f'in_band_{model_key}'] = [
        lo <= int(n) <= hi for (lo, hi), n in zip(low_high, compressed_df[f'tokens_{model_key}'])
    ]

# 결과 확인
new_failed_mask = (~compressed_df['in_band_gpt2']) | (~compressed_df['in_band_pythia-160m'])
new_failed_df = compressed_df[new_failed_mask]

print(f"자동 압축 적용 후 총 실패 항목 수: {len(new_failed_df)}")
if len(new_failed_df) > 0:
    display(new_failed_df[['topic_id', 'length_tier', 'style', 'tokens_gpt2', 'tokens_pythia-160m']].head(10))
else:
    print("🎉 축하합니다! 모든 400개 candidate 텍스트가 양쪽 토크나이저의 목표 대역(Band) 조건을 충족합니다!")

자동 압축 적용 후 총 실패 항목 수: 10


,topic_id,length_tier,style,tokens_gpt2,tokens_pythia-160m
7,T002,short,code_plus_natural_language,71,71
27,T007,short,code_plus_natural_language,71,71
127,T032,short,code_plus_natural_language,78,78
155,T039,medium,code_plus_natural_language,139,116
163,T041,medium,code_plus_natural_language,139,116
191,T048,medium,code_plus_natural_language,131,108
267,T067,medium,code_plus_natural_language,131,108
339,T085,long,code_plus_natural_language,158,122
351,T088,long,code_plus_natural_language,150,114
359,T090,long,code_plus_natural_language,158,122


## 5. Pre-registered analysis outline (after freeze only)

**Core observables per text × model × block**
- `update_norm_per_token = ||h_(l+1) - h_l||_F / sqrt(T)` (keep raw Frobenius norm as QC).
- `input_update_cosine = cos(h_l, Δh_l)`; compute from tensors when available.
- `output_input_norm_ratio = ||h_(l+1)||_F / ||h_l||_F`.

**Primary exploratory-confirmatory contrast framework**
- Keep GPT-2 and Pythia analyses separate first, then treat model as a factor only in a prespecified joint model because hidden-state scales differ.
- Within each `topic_id`, compare styles while controlling actual tokenizer-specific token count and length tier.
- Layer depth is aligned by normalized block position `rho = block_idx / max(L - 1, 1)` for block indices. Preserve raw block index too.
- Use a topic-cluster bootstrap with 10,000 replicates, resampling 100 topic IDs and retaining all four style variants inside each selected topic.
- For layer-wise familywise inference, use a preregistered max-statistic permutation or explicitly limit confirmatory claims to prespecified layer summaries; use Holm correction across the three core observables if testing each as a co-primary outcome.
- Do not claim chaos, physical energy, thermodynamic temperature, or a phase transition from these geometric observables alone.

The N02 pilot baseline is in the separate file `N02_Pilot_Baseline_v1.0.md`; it remains exploratory and is not part of the Main Evaluation dataset.
